# 14 · Reproducible supplementary material and manuscript audit

**Purpose.** Consolidate already validated outputs from Gates 8 and 10–14 into
journal-ready supplementary tables and a standalone LaTeX supplement.

This notebook performs **no model re-estimation** and uses no raw microdata. It verifies
that every reported sensitivity result comes from a passed analytical gate, separates
primary inference from robustness analysis, and exports a text artifact bundle that can
be committed reproducibly.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil

import numpy as np
import pandas as pd

ROOT = Path("/content/colombia-stochastic-access")
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
OUT = ROOT / "outputs/bogota_em2023/gate15"
PAPER = ROOT / "paper"
SUPPLEMENT = PAPER / "supplementary"
TABLES = SUPPLEMENT / "tables"

for directory in (OUT, SUPPLEMENT, TABLES):
    directory.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Environment outside Colab or Drive not mounted: {exc}")

G8 = Path(os.environ.get("GATE8_DIR_OVERRIDE", DRIVE_BASE / "results/gate8"))
G10 = Path(os.environ.get("GATE10_DIR_OVERRIDE", DRIVE_BASE / "results/gate10"))
G11 = Path(os.environ.get("GATE11_DIR_OVERRIDE", DRIVE_BASE / "results/gate11"))
G12 = Path(os.environ.get("GATE12_DIR_OVERRIDE", DRIVE_BASE / "results/gate12"))
G13 = Path(os.environ.get("GATE13_DIR_OVERRIDE", DRIVE_BASE / "results/gate13"))
G14 = Path(os.environ.get("GATE14_DIR_OVERRIDE", DRIVE_BASE / "results/gate14"))

paths = {
    "gate8": G8 / "gate_8.json",
    "tau": G8 / "tau_sensitivity.csv",
    "gate10": G10 / "gate_10.json",
    "robust_point": G10 / "robustness_female_male_point_estimates.csv",
    "robust_intervals": G10 / "robustness_sex_bootstrap_intervals.csv",
    "robust_stability": G10 / "robustness_sex_stability_summary.csv",
    "gate11": G11 / "gate_11.json",
    "primary_intervals": G11 / "final_female_male_intervals.csv",
    "stratum": G11 / "final_stratum_spearman_intervals.csv",
    "gate12": G12 / "gate_12.json",
    "district_support": G12 / "bogota_dc_internal_scope_support.csv",
    "district_targets": G12 / "bogota_dc_internal_target_summary.csv",
    "municipal": G12 / "region_municipality_origin_contributions.csv",
    "gate13": G13 / "gate_13.json",
    "paired": G13 / "paired_scope_contrast_intervals.csv",
    "gate14": G14 / "gate_14.json",
    "main_table": G14 / "table_1_main_scope_sex_gaps.csv",
    "amplification_table": G14 / "table_2_paired_scope_amplification.csv",
    "municipal_table": G14 / "table_3_top_external_municipality_contributions.csv",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing validated inputs: " + ", ".join(missing))

gate_paths = {8: paths["gate8"], 10: paths["gate10"], 11: paths["gate11"],
              12: paths["gate12"], 13: paths["gate13"], 14: paths["gate14"]}
gates = {number: json.loads(path.read_text(encoding="utf-8"))
         for number, path in gate_paths.items()}
for number, report in gates.items():
    status = report.get("gate", report)
    if not bool(status.get(f"gate_{number}_passed", False)):
        raise RuntimeError(f"Gate {number} is not approved.")

print("Validated dependencies:", ", ".join(f"Gate {number}" for number in gates))
print("Output:", OUT)


## 1. Load frozen analytical outputs

The primary estimand uses the full Bogotá Region, core-50 purpose targets,
outside-target residential conditioning, and regularization `tau=20`.
Robustness results vary the reference population, target coverage, and conditioning.


In [ ]:
tau = pd.read_csv(paths["tau"])
robust_point = pd.read_csv(paths["robust_point"])
robust_intervals = pd.read_csv(paths["robust_intervals"])
robust_stability = pd.read_csv(paths["robust_stability"])
primary_intervals = pd.read_csv(paths["primary_intervals"])
stratum = pd.read_csv(paths["stratum"])
district_support = pd.read_csv(paths["district_support"])
district_targets = pd.read_csv(paths["district_targets"])
municipal = pd.read_csv(paths["municipal"])
paired = pd.read_csv(paths["paired"])
main_table = pd.read_csv(paths["main_table"])
amplification_table = pd.read_csv(paths["amplification_table"])
municipal_table = pd.read_csv(paths["municipal_table"])

purpose_labels = {
    "employment": "Employment",
    "education": "Education",
    "health": "Health",
}
scheme_labels = {
    "full_population": "Full population",
    "valid_dimension": "Valid sex categories",
}
target_labels = {
    "core_50": "Core 50%",
    "extended_80": "Extended 80%",
}
conditioning_labels = {
    "outside_target": "Outside target",
    "all_residences": "All residences",
}
effect_labels = {
    "female_minus_male_total": "Total female-minus-male gap",
    "residential_contribution": "Residential component",
    "kernel_contribution": "Mobility-kernel component",
    "region_minus_city__female_minus_male_total": "Total gap",
    "region_minus_city__residential_contribution": "Residential component",
    "region_minus_city__kernel_contribution": "Mobility-kernel component",
    "relative_attenuation": "Relative attenuation",
}

print("Rows loaded:")
for name, frame in {
    "tau sensitivity": tau,
    "robustness grid": robust_point,
    "primary intervals": primary_intervals,
    "paired intervals": paired,
}.items():
    print(f"  {name}: {len(frame)}")


## 2. Build supplementary tables

In [ ]:
def write_latex(frame, filename, caption, label, longtable=False, float_format="%.3f"):
    path = TABLES / filename
    latex = frame.to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label,
        float_format=float_format,
    )
    path.write_text(latex, encoding="utf-8")
    return path

# Table S1: tau sensitivity
tau_summary = (
    tau.groupby("tau", observed=True)
    .agg(
        maximum_absolute_relative_change=(
            "relative_change_from_tau20", lambda values: values.abs().max()
        ),
        minimum_spearman_vs_tau20=("spearman_vs_tau20", "min"),
        median_spearman_vs_tau20=("spearman_vs_tau20", "median"),
    )
    .reset_index()
    .rename(columns={
        "tau": "Tau",
        "maximum_absolute_relative_change": "Maximum absolute relative change",
        "minimum_spearman_vs_tau20": "Minimum Spearman correlation",
        "median_spearman_vs_tau20": "Median Spearman correlation",
    })
)
write_latex(
    tau_summary, "table_s1_tau_sensitivity.tex",
    "Sensitivity of group FPT estimates to the regularization parameter.",
    "tab:s1_tau",
)

# Table S2: range over the 24 robustness specifications
stability_table = robust_stability.copy()
stability_table["Purpose"] = stability_table["purpose_group"].map(purpose_labels)
stability_table = stability_table[[
    "Purpose", "minimum_total_gap", "maximum_total_gap",
    "minimum_residential_contribution", "maximum_residential_contribution",
    "minimum_kernel_contribution", "maximum_kernel_contribution",
]].rename(columns={
    "minimum_total_gap": "Minimum total gap",
    "maximum_total_gap": "Maximum total gap",
    "minimum_residential_contribution": "Minimum residential",
    "maximum_residential_contribution": "Maximum residential",
    "minimum_kernel_contribution": "Minimum kernel",
    "maximum_kernel_contribution": "Maximum kernel",
})
write_latex(
    stability_table, "table_s2_robustness_ranges.tex",
    "Range of female-minus-male decomposition estimates over 24 prespecified robustness specifications.",
    "tab:s2_ranges",
)

# Table S3: complete 24-specification grid
grid_table = robust_point.copy()
grid_table["Reference"] = grid_table["reference_scheme"].map(scheme_labels)
grid_table["Target"] = grid_table["target_definition"].map(target_labels)
grid_table["Conditioning"] = grid_table["conditioning"].map(conditioning_labels)
grid_table["Purpose"] = grid_table["purpose_group"].map(purpose_labels)
grid_table = grid_table[[
    "Reference", "Target", "Conditioning", "Purpose",
    "female_minus_male_total", "residential_contribution", "kernel_contribution",
]].rename(columns={
    "female_minus_male_total": "Total gap",
    "residential_contribution": "Residential",
    "kernel_contribution": "Kernel",
})
write_latex(
    grid_table, "table_s3_complete_robustness_grid.tex",
    "Female-minus-male decomposition for the complete prespecified robustness grid.",
    "tab:s3_grid", longtable=True,
)

# Table S4: final regional inference
primary_table = primary_intervals.copy()
primary_table["Purpose"] = primary_table["purpose_group"].map(purpose_labels)
primary_table["Effect"] = primary_table["effect"].map(effect_labels)
primary_table = primary_table[[
    "Purpose", "Effect", "point_estimate", "bootstrap_sd",
    "ci_lower", "ci_upper", "simultaneous_ci_lower", "simultaneous_ci_upper",
]].rename(columns={
    "point_estimate": "Estimate",
    "bootstrap_sd": "Bootstrap SD",
    "ci_lower": "Pointwise lower",
    "ci_upper": "Pointwise upper",
    "simultaneous_ci_lower": "Simultaneous lower",
    "simultaneous_ci_upper": "Simultaneous upper",
})
write_latex(
    primary_table, "table_s4_regional_inference.tex",
    "Final Bogotá-Region household-bootstrap inference.",
    "tab:s4_region",
)

# Table S5: district network support
support_table = district_support.melt(var_name="Metric", value_name="Value")
support_table["Metric"] = support_table["Metric"].str.replace("_", " ").str.capitalize()
write_latex(
    support_table, "table_s5_district_support.tex",
    "Support and strongly connected component diagnostics for the Bogotá D.C.-internal network.",
    "tab:s5_support",
)

# Table S6: district target sets
targets_table = district_targets.copy()
targets_table["Purpose"] = targets_table["purpose_group"].map(purpose_labels)
targets_table = targets_table[[
    "Purpose", "target_definition", "target_nodes", "selected_supported_share"
]].rename(columns={
    "target_definition": "Definition",
    "target_nodes": "Target nodes",
    "selected_supported_share": "Supported arrival share",
})
write_latex(
    targets_table, "table_s6_district_targets.tex",
    "Scope-specific Bogotá D.C.-internal opportunity target sets.",
    "tab:s6_targets",
)

# Table S7: paired scope inference
paired_table = paired.copy()
paired_table["Purpose"] = paired_table["purpose_group"].map(purpose_labels)
paired_table["Effect"] = paired_table["effect"].map(effect_labels)
paired_table["Scale"] = np.where(paired_table["effect"].eq("relative_attenuation"), 100.0, 1.0)
for column in ["point_estimate", "bootstrap_sd", "simultaneous_ci_lower", "simultaneous_ci_upper"]:
    paired_table[column] = paired_table[column] * paired_table["Scale"]
paired_table["Unit"] = np.where(
    paired_table["effect"].eq("relative_attenuation"), "Percent", "FPT steps"
)
paired_table = paired_table[[
    "Purpose", "Effect", "Unit", "point_estimate", "bootstrap_sd",
    "simultaneous_ci_lower", "simultaneous_ci_upper",
]].rename(columns={
    "point_estimate": "Estimate",
    "bootstrap_sd": "Bootstrap SD",
    "simultaneous_ci_lower": "Simultaneous lower",
    "simultaneous_ci_upper": "Simultaneous upper",
})
write_latex(
    paired_table, "table_s7_paired_scope_inference.tex",
    "Paired Bogotá-Region minus Bogotá D.C.-internal scope contrasts.",
    "tab:s7_paired",
)

# Table S8: stratum association
stratum_table = stratum.copy()
stratum_table["Purpose"] = stratum_table["purpose_group"].map(purpose_labels)
stratum_table = stratum_table[[
    "Purpose", "point_estimate", "bootstrap_sd", "ci_lower", "ci_upper"
]].rename(columns={
    "point_estimate": "Spearman rho",
    "bootstrap_sd": "Bootstrap SD",
    "ci_lower": "Lower 95% CI",
    "ci_upper": "Upper 95% CI",
})
write_latex(
    stratum_table, "table_s8_stratum_association.tex",
    "Association between socioeconomic stratum and observed mean FPT.",
    "tab:s8_stratum",
)

display(tau_summary.round(3))
display(stability_table.round(3))
display(paired_table.round(3))


## 3. Write the standalone supplementary document

In [ ]:
supplement_text = r"""\documentclass[11pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{lmodern}
\usepackage{booktabs,longtable,graphicx,microtype}
\usepackage{setspace,lineno,hyperref}
\doublespacing
\linenumbers

\title{Supplementary material\\
Metropolitan scope amplifies measured gender disparities in stochastic urban accessibility}
\author{Andy Rafael Domínguez-Monterroza}
\date{}

\begin{document}
\maketitle

\section{Supplementary methods}

\subsection{Interpretation of the stochastic-accessibility estimand}
The origin--destination Markov chain represents aggregate survey-weighted population
flows. A first-passage step is a dimensionless transition in this fitted flow system and
must not be interpreted as a minute of travel or as an observed individual's itinerary.
For every purpose, residential distributions are conditioned on starting outside the
corresponding target set in the primary specification.

\subsection{Regularization sensitivity}
Group-specific rows are shrunk toward the population kernel using
$\omega_{ig}=\mathrm{ESS}_{ig}/(\mathrm{ESS}_{ig}+\tau)$. The primary value is
$\tau=20$; values 10 and 50 were prespecified sensitivity checks. Table~\ref{tab:s1_tau}
summarizes the maximum magnitude change and the minimum and median rank correlation of
origin-specific FPT values relative to $\tau=20$.

\input{supplementary/tables/table_s1_tau_sensitivity.tex}

\subsection{Specification grid}
The robustness grid crosses two reference populations (all survey-valid observations or
only observations with valid categories for the analysed dimension), two target
definitions (core 50\% and extended 80\% of supported weighted arrivals), and two
residential conventions (outside-target conditioning or all residences). Across three
purposes this yields 24 sex-gap specifications. These checks assess specification
dependence; they were not used to select the primary result.

\input{supplementary/tables/table_s2_robustness_ranges.tex}
\input{supplementary/tables/table_s3_complete_robustness_grid.tex}

\subsection{Bootstrap inference}
The final regional analysis uses 500 joint Poisson(1) household-cluster bootstrap
replicates. The same multiplier is assigned to every record from a household and is
propagated jointly to person and trip weights. Simultaneous intervals use a bootstrap
max-$t$ critical value over the three destination purposes within each effect family.
The paired scope analysis likewise uses 500 shared household multipliers in the regional
and district systems. The earlier 50-replicate specification grid and 200-replicate
standalone district analysis are robustness diagnostics rather than the final paired
inference.

\section{Supplementary results}

\subsection{Regional inference}
\input{supplementary/tables/table_s4_regional_inference.tex}

\subsection{District support and target reconstruction}
The Bogotá D.C.-internal analysis retains the largest directed strongly connected
component. Purpose-specific targets are reconstructed within the district scope rather
than inherited from the regional system.

\input{supplementary/tables/table_s5_district_support.tex}
\input{supplementary/tables/table_s6_district_targets.tex}

\subsection{Paired territorial-scope contrasts}
Positive step-valued contrasts indicate a larger female-minus-male gap in the regional
system. Relative attenuation is reported in percentage points. The scope comparison is
descriptive and includes both network truncation and scope-specific opportunity
geography.

\input{supplementary/tables/table_s7_paired_scope_inference.tex}

\subsection{Socioeconomic-stratum diagnostic}
\input{supplementary/tables/table_s8_stratum_association.tex}

\subsection{External-municipality attribution}
Figure~\ref{fig:s1_municipal} reports additive origin-level contributions to the fitted
regional mobility-kernel component. These values are not causal deletion effects and
must not be interpreted as effects of removing or administratively integrating a
municipality.

\begin{figure}[ht]
\centering
\includegraphics[width=0.96\textwidth]{figures/figure_3_external_municipality_kernel_contributions.png}
\caption{External-municipality contributions to the regional mobility-kernel component.}
\label{fig:s1_municipal}
\end{figure}

\section{Reproducibility and interpretation boundaries}
All supplementary tables are generated from passed analytical gates without
re-estimating the models. The analysis is cross-sectional and descriptive. Survey
weights and clustered bootstrap inference propagate sampling uncertainty but do not
remove recall error, nonresponse bias, unobserved modal and temporal heterogeneity, or
the limited gender categories in the processed survey.

\end{document}
"""
supplement_path = PAPER / "supplementary_material.tex"
supplement_path.write_text(supplement_text, encoding="utf-8")
print("Wrote:", supplement_path)


## 4. Gate 15: supplementary-package consistency

In [ ]:
expected_grid_rows = 2 * 2 * 2 * 3
generated_tables = sorted(TABLES.glob("table_s*.tex"))

checks = {
    "gate8_dependency_valid": True,
    "gate10_dependency_valid": True,
    "gate11_dependency_valid": True,
    "gate12_dependency_valid": True,
    "gate13_dependency_valid": True,
    "gate14_dependency_valid": True,
    "tau_grid_complete": set(pd.to_numeric(tau["tau"])) == {10.0, 20.0, 50.0},
    "robustness_grid_complete": len(robust_point) == expected_grid_rows,
    "robustness_total_gap_positive_all_specifications": bool(
        robust_point["female_minus_male_total"].gt(0).all()
    ),
    "robustness_kernel_positive_all_specifications": bool(
        robust_point["kernel_contribution"].gt(0).all()
    ),
    "primary_interval_family_complete": len(primary_intervals) == 9,
    "paired_interval_family_complete": len(paired) == 12,
    "main_table_complete": len(main_table) == 6,
    "main_amplification_table_complete": len(amplification_table) == 12,
    "supplementary_tables_complete": len(generated_tables) == 8,
    "supplementary_source_exists": supplement_path.exists(),
    "all_exported_text_nonempty": bool(
        supplement_path.stat().st_size > 0
        and all(path.stat().st_size > 0 for path in generated_tables)
    ),
}
checks["gate_15_passed"] = bool(all(checks.values()))

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "frozen supplementary package and manuscript consistency audit",
    "dependencies": sorted(gates),
    "primary_specification": {
        "scope": "Bogotá Region",
        "target_definition": "core_50",
        "conditioning": "outside_target",
        "regularization_tau": 20,
    },
    "robustness_grid": {
        "reference_schemes": 2,
        "target_definitions": 2,
        "conditioning_modes": 2,
        "purposes": 3,
        "rows": int(len(robust_point)),
    },
    "generated_files": [
        str(supplement_path.relative_to(ROOT)),
        *[str(path.relative_to(ROOT)) for path in generated_tables],
    ],
    "gate": checks,
}
(OUT / "gate_15.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
pd.Series(checks, name="result").to_csv(OUT / "gate_15_checks.csv")
display(pd.Series(checks, name="result"))

persistent = DRIVE_BASE / "results/gate15"
persistent.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent / artifact.name)

print("Gate 15:", "PASSED" if checks["gate_15_passed"] else "FAILED")
print("Persistent copy:", persistent)


## 5. Export text artifact bundle

The final JSON object contains the exact generated LaTeX files so that they can be
recovered from the executed notebook and committed without relying on Colab's handling
of non-notebook files.


In [ ]:
artifact_paths = [supplement_path, *generated_tables]
artifact_bundle = {
    str(path.relative_to(ROOT)): path.read_text(encoding="utf-8")
    for path in artifact_paths
}
print("ARTIFACT_BUNDLE_BEGIN")
print(json.dumps(artifact_bundle, ensure_ascii=False))
print("ARTIFACT_BUNDLE_END")
